# Online Retail — Optimized Modeling

این نوت‌بوک روی دو دیتاست آماده‌شده پروژه اجرا می‌شود. مسیر فایل‌ها به‌صورت خودکار از پوشه نوت‌بوک و زیرپوشه‌های آن پیدا می‌شود؛ بنابراین وابسته به مسیر ثابت ویندوز، Colab یا سیستم خاصی نیست.

**هدف‌ها:** Customer Segmentation، ارزش اقتصادی و فرصت بازاریابی، انتخاب ۵۰۰ مشتری هدف و Association Rule Mining دوآیتمی. این پروژه شامل پیش‌بینی Churn نیست.

In [1]:
# 1) Imports and robust project-file discovery
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

RANDOM_STATE = 42
OUTPUT_DIR = Path("modeling_outputs")

def find_project_file(filename, start=None):
    """Find a unique project input file recursively, preferring prepared_data folders."""
    start = Path(start or Path.cwd()).resolve()
    search_roots = [start] + list(start.parents[:3])
    matches = []
    seen = set()
    for root in search_roots:
        if not root.exists():
            continue
        # Search project tree but ignore version-control and notebook checkpoints.
        for p in root.rglob(filename):
            if any(part in {'.git', '.ipynb_checkpoints', '__pycache__'} for part in p.parts):
                continue
            rp = p.resolve()
            if rp not in seen:
                seen.add(rp); matches.append(rp)
        if matches:
            break
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename!r}. Started at: {start}. "
            "Place it in ./prepared_data/ or a project subfolder."
        )
    matches.sort(key=lambda p: (0 if p.parent.name.lower() == 'prepared_data' else 1, len(p.parts), str(p)))
    best_rank = (0 if matches[0].parent.name.lower() == 'prepared_data' else 1, len(matches[0].parts))
    equally_ranked = [p for p in matches if (0 if p.parent.name.lower() == 'prepared_data' else 1, len(p.parts)) == best_rank]
    if len(equally_ranked) > 1:
        raise RuntimeError("Ambiguous input file. Multiple equally preferred copies found:\\n" +
                           "\\n".join(map(str, equally_ranked)))
    return matches[0]

customer_path = find_project_file("customer_analysis.csv")
basket_path = find_project_file("product_basket.csv.gz")
print("Customer file:", customer_path)
print("Basket file:", basket_path)

Customer file: C:\Users\Zhivar\Desktop\OnlineShope_DataMining\002.DataPrepration\prepared_data\customer_analysis.csv
Basket file: C:\Users\Zhivar\Desktop\OnlineShope_DataMining\002.DataPrepration\prepared_data\product_basket.csv.gz


In [2]:
# 2) Load and validate datasets
customers = pd.read_csv(customer_path, low_memory=False)
baskets = pd.read_csv(basket_path, compression="infer", low_memory=False)

required_customer = {"CustomerID", "Recency", "Frequency", "Monetary", "NetMonetary",
                     "AvgOrderValue", "UniqueProducts", "PurchasedInLast90Days", "SpendGrowthIndex"}
missing = required_customer - set(customers.columns)
if missing:
    raise ValueError(f"customer_analysis.csv is missing required columns: {sorted(missing)}")
if "InvoiceNo" not in baskets.columns or baskets.shape[1] < 3:
    raise ValueError("product_basket.csv.gz must contain InvoiceNo and at least two product columns.")
if customers.empty or baskets.empty:
    raise ValueError("One or both input datasets are empty.")
if customers["CustomerID"].isna().any():
    raise ValueError("CustomerID contains missing values; resolve this in Data Preparation.")

seg_features = ["Seg_Recency", "Seg_Frequency", "Seg_Monetary", "Seg_AvgOrderValue",
                "Seg_UniqueProducts", "Seg_AvgQuantityPerInvoice"]
missing_seg = [c for c in seg_features if c not in customers.columns]
if missing_seg:
    raise ValueError(f"Missing segmentation features: {missing_seg}")
if customers[seg_features].isna().any().any():
    raise ValueError("Segmentation features contain missing values.")
print(f"Loaded {len(customers):,} customers and {len(baskets):,} invoices "
      f"with {baskets.shape[1]-1:,} product columns.")

Loaded 4,334 customers and 18,267 invoices with 1,612 product columns.


## 3) Feature redundancy check and model comparison

در نسخه قبلی، شش ویژگی وارد K-Means شده بودند. دو ویژگی `Seg_AvgOrderValue` و `Seg_AvgQuantityPerInvoice` همبستگی بالایی دارند (حدود 0.82). این دو ویژگی را در آزمایش حذف می‌کنیم تا از وزن‌دهی تکراری به الگوی مشابه جلوگیری شود. ویژگی‌های `Seg_` از قبل در مرحله Data Preparation مقیاس‌بندی شده‌اند؛ دوباره آن‌ها را Scale نمی‌کنیم.

In [3]:
# 3) Correlation audit and compare KMeans / Ward / GMM
corr = customers[seg_features].corr()
display(corr.round(3))

feature_sets = {
    "all_6_features": seg_features,
    "reduced_4_features": ["Seg_Recency", "Seg_Frequency", "Seg_Monetary", "Seg_UniqueProducts"],
}
rng = np.random.RandomState(RANDOM_STATE)
sample_idx = rng.choice(len(customers), size=min(2500, len(customers)), replace=False)
model_rows = []

for set_name, cols in feature_sets.items():
    X = customers[cols].to_numpy(dtype=float)
    for k in range(2, 9):
        model = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=30)
        labels = model.fit_predict(X)
        sample_sil = silhouette_score(X[sample_idx], labels[sample_idx])
        model_rows.append({"features": set_name, "method": "KMeans", "k": k,
                           "silhouette": sample_sil,
                           "davies_bouldin": davies_bouldin_score(X, labels),
                           "calinski_harabasz": calinski_harabasz_score(X, labels),
                           "min_cluster_size": int(np.bincount(labels).min())})
    if set_name == "reduced_4_features":
        for method_name, estimator in [
            ("Ward", lambda k: AgglomerativeClustering(n_clusters=k, linkage="ward")),
            ("GMM", lambda k: GaussianMixture(n_components=k, random_state=RANDOM_STATE,
                                               n_init=3, reg_covar=1e-5))
        ]:
            for k in range(2, 7):
                model = estimator(k)
                labels = model.fit_predict(X)
                model_rows.append({"features": set_name, "method": method_name, "k": k,
                                   "silhouette": silhouette_score(X[sample_idx], labels[sample_idx]),
                                   "davies_bouldin": davies_bouldin_score(X, labels),
                                   "calinski_harabasz": calinski_harabasz_score(X, labels),
                                   "min_cluster_size": int(np.bincount(labels).min())})

model_comparison = pd.DataFrame(model_rows).sort_values(
    ["silhouette", "davies_bouldin"], ascending=[False, True])
display(model_comparison.head(15).round(4))

,Seg_Recency,Seg_Frequency,Seg_Monetary,Seg_AvgOrderValue,Seg_UniqueProducts,Seg_AvgQuantityPerInvoice
Seg_Recency,1.000,-0.571,-0.486,-0.124,-0.444,-0.162
Seg_Frequency,-0.571,1.000,0.808,0.162,0.621,0.149
Seg_Monetary,-0.486,0.808,1.000,0.708,0.694,0.596
Seg_AvgOrderValue,-0.124,0.162,0.708,1.000,0.408,0.825
Seg_UniqueProducts,-0.444,0.621,0.694,0.408,1.000,0.428
Seg_AvgQuantityPerInvoice,-0.162,0.149,0.596,0.825,0.428,1.000


,features,method,k,silhouette,davies_bouldin,calinski_harabasz,min_cluster_size
7,reduced_4_features,KMeans,2,0.3989,0.9516,3902.8560,1752
14,reduced_4_features,Ward,2,0.3889,0.9706,3663.7474,1635
0,all_6_features,KMeans,2,0.3104,1.1953,2491.7557,2095
8,reduced_4_features,KMeans,3,0.2965,1.1384,3136.9417,890
9,reduced_4_features,KMeans,4,0.2858,1.1975,2637.5622,824
19,reduced_4_features,GMM,2,0.2611,1.1576,2107.8631,1505
10,reduced_4_features,KMeans,5,0.2495,1.1732,2464.5936,493
1,all_6_features,KMeans,3,0.2472,1.2687,2078.8748,1288
15,reduced_4_features,Ward,3,0.2326,1.3543,2500.3525,1033
12,reduced_4_features,KMeans,7,0.2305,1.1873,2120.6987,281


In [4]:
# 4) Fit selected model on all customers
# Selection based on internal cluster metrics plus interpretability.
final_features = ["Seg_Recency", "Seg_Frequency", "Seg_Monetary", "Seg_UniqueProducts"]
X_final = customers[final_features].to_numpy(dtype=float)
final_model = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=30)
customers["Cluster"] = final_model.fit_predict(X_final)

final_silhouette = silhouette_score(X_final[sample_idx],
                                    customers["Cluster"].to_numpy()[sample_idx])
final_db = davies_bouldin_score(X_final, customers["Cluster"])
final_ch = calinski_harabasz_score(X_final, customers["Cluster"])

cluster_profile = customers.groupby("Cluster").agg(
    Customers=("CustomerID", "nunique"),
    MedianRecency=("Recency", "median"),
    MedianFrequency=("Frequency", "median"),
    MedianMonetary=("Monetary", "median"),
    MedianNetMonetary=("NetMonetary", "median"),
    MedianUniqueProducts=("UniqueProducts", "median"),
    MeanRecency=("Recency", "mean"),
    MeanFrequency=("Frequency", "mean"),
    MeanNetMonetary=("NetMonetary", "mean")
).round(2)
display(cluster_profile)
print(f"Final KMeans: k=2 | sampled Silhouette={final_silhouette:.4f} | "
      f"Davies-Bouldin={final_db:.4f} | Calinski-Harabasz={final_ch:.2f}")

,Customers,MedianRecency,MedianFrequency,MedianMonetary,MedianNetMonetary,MedianUniqueProducts,MeanRecency,MeanFrequency,MeanNetMonetary
Cluster,,,,,,,,,
0,2582,99.0,1.0,344.95,338.83,20.0,135.83,1.63,470.01
1,1752,18.0,5.0,1936.20,1891.83,88.0,30.45,8.10,4038.22


Final KMeans: k=2 | sampled Silhouette=0.3989 | Davies-Bouldin=0.9516 | Calinski-Harabasz=3902.86


## 5) Customer economic value, opportunity and top-500 campaign list

این امتیازها شاخص‌های اولویت‌بندی داخلی پروژه‌اند، نه احتمال خرید یا خروجی یک مدل پیش‌بینی‌شده. مشتریان دارای NetMonetary مثبت برای لیست ۵۰۰ نفره در اولویت‌اند.

In [5]:
# 5) Scoring and campaign targeting
def percentile(s):
    return s.rank(pct=True).fillna(0)

customers["EconomicScore"] = (
    0.45 * percentile(customers["NetMonetary"]) +
    0.35 * percentile(customers["Frequency"]) +
    0.20 * percentile(customers["AvgOrderValue"])
) * 100

growth = customers["SpendGrowthIndex"].replace([np.inf, -np.inf], np.nan).clip(-1, 1)
growth_norm = ((growth + 1) / 2).fillna(0.5)
customers["OpportunityScore"] = (
    0.45 * percentile(-customers["Recency"]) +
    0.30 * growth_norm +
    0.25 * customers["PurchasedInLast90Days"].fillna(0)
) * 100

customers["PriorityScore"] = (
    0.50 * customers["EconomicScore"] +
    0.35 * customers["OpportunityScore"] +
    0.15 * percentile(-customers["Recency"]) * 100
)

positive_value = customers[customers["NetMonetary"] > 0].copy()
target_pool = positive_value if len(positive_value) >= 500 else customers.copy()
top500 = target_pool.nlargest(500, "PriorityScore").copy()

def campaign_recommendation(row):
    if row["EconomicScore"] >= 80 and row["Recency"] <= 30:
        return "VIP loyalty / early access"
    if row["EconomicScore"] >= 80:
        return "VIP win-back / personal offer"
    if row["Recency"] <= 30 and row["Frequency"] >= 2:
        return "Cross-sell / bundle"
    if row["Recency"] > 90:
        return "Reactivation offer"
    return "Personalized product recommendation"

top500["Campaign"] = top500.apply(campaign_recommendation, axis=1)
print("Selected customers:", len(top500), "| unique IDs:", top500["CustomerID"].nunique())
display(top500[["CustomerID", "Cluster", "EconomicScore", "OpportunityScore",
                "PriorityScore", "Campaign"]].head(10))

Selected customers: 500 | unique IDs: 500


,CustomerID,Cluster,EconomicScore,OpportunityScore,PriorityScore,Campaign
1058,13777,1,97.998962,97.763580,98.157897,VIP loyalty / early access
2836,16210,1,97.952238,97.232255,97.709762,VIP loyalty / early access
3680,17389,1,98.540032,94.907159,97.428685,VIP loyalty / early access
4197,18102,1,99.867328,92.065764,97.097844,VIP loyalty / early access
2461,15694,1,92.833410,97.200376,95.378000,VIP loyalty / early access
1689,14646,1,99.865021,87.707269,95.332408,VIP loyalty / early access
1983,15061,1,99.118597,90.329821,95.228151,VIP loyalty / early access
2063,15159,1,96.987771,91.484852,95.215937,VIP loyalty / early access
570,13098,1,98.546954,88.560423,94.971979,VIP loyalty / early access
3174,16684,1,99.387979,89.829216,94.862295,VIP loyalty / early access


## 6) Association Rule Mining — frequent 2-item rules

برای جلوگیری از وابستگی به نصب پکیج خارجی، قواعد دوآیتمی با شمارش دقیق هم‌رخدادی محاسبه می‌شوند. این بخش **FP-Growth نیست** و قواعد چندآیتمی را استخراج نمی‌کند. آستانه‌ها: support ≥ 0.5%، confidence ≥ 30% و lift ≥ 1.10.

In [6]:
# 6) Exact pairwise co-occurrence rules
product_cols = [c for c in baskets.columns if c != "InvoiceNo"]
binary = baskets[product_cols].fillna(0).to_numpy(dtype=np.uint8)
if not np.isin(binary, [0, 1]).all():
    raise ValueError("Basket product columns must be binary (0/1).")

min_product_support = 0.005
product_support_all = binary.mean(axis=0)
eligible = np.where(product_support_all >= min_product_support)[0]
# Limit to 150 frequent products for transparent, bounded pairwise computation.
eligible = eligible[np.argsort(product_support_all[eligible])[::-1][:150]]
selected_products = [product_cols[i] for i in eligible]
B = baskets[selected_products].fillna(0).to_numpy(dtype=np.uint8)
n_invoices = len(B)
item_support = B.mean(axis=0)
co_support = (B.T @ B) / n_invoices

rule_rows = []
for i, antecedent in enumerate(selected_products):
    for j, consequent in enumerate(selected_products):
        if i == j:
            continue
        support = float(co_support[i, j])
        confidence = support / float(item_support[i])
        lift = confidence / float(item_support[j])
        if support >= 0.005 and confidence >= 0.30 and lift >= 1.10:
            rule_rows.append({
                "antecedent": str(antecedent), "consequent": str(consequent),
                "support": support, "confidence": confidence,
                "consequent_support": float(item_support[j]), "lift": lift
            })
association_rules = pd.DataFrame(rule_rows, columns=[
    "antecedent", "consequent", "support", "confidence", "consequent_support", "lift"
])
if not association_rules.empty:
    association_rules = association_rules.sort_values(
        ["lift", "confidence", "support"], ascending=False).reset_index(drop=True)
print(f"Products evaluated: {len(selected_products)} | Rules found: {len(association_rules)}")
display(association_rules.head(20).round(4))

Products evaluated: 150 | Rules found: 142


,antecedent,consequent,support,confidence,consequent_support,lift
0,22553,22557,0.0108,0.3518,0.0310,11.3535
1,22557,22553,0.0108,0.3481,0.0307,11.3535
2,21975,84992,0.0123,0.3750,0.0337,11.1384
3,84992,21975,0.0123,0.3659,0.0328,11.1384
4,22553,22554,0.0128,0.4161,0.0385,10.7960
5,22554,22553,0.0128,0.3310,0.0307,10.7960
6,22554,22557,0.0129,0.3338,0.0310,10.7732
7,22557,22554,0.0129,0.4152,0.0385,10.7732
8,23204,22356,0.0136,0.4271,0.0406,10.5146
9,22356,23204,0.0136,0.3356,0.0319,10.5146


In [7]:
# 7) Export all outputs and metrics
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
customers.to_csv(OUTPUT_DIR / "customer_segments_and_scores.csv", index=False)
cluster_profile.to_csv(OUTPUT_DIR / "customer_segment_profile.csv")
top500.to_csv(OUTPUT_DIR / "top500_customers.csv", index=False)
association_rules.to_csv(OUTPUT_DIR / "association_rules_2item.csv", index=False)
model_comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

metrics = {
    "customers": int(len(customers)),
    "invoices": int(len(baskets)),
    "product_columns": int(len(product_cols)),
    "selected_model": "KMeans",
    "selected_k": 2,
    "selected_features": final_features,
    "sample_size_for_silhouette": int(len(sample_idx)),
    "silhouette_sampled": float(final_silhouette),
    "davies_bouldin": float(final_db),
    "calinski_harabasz": float(final_ch),
    "cluster_sizes": {str(k): int(v) for k, v in customers["Cluster"].value_counts().sort_index().items()},
    "top500_unique_customers": int(top500["CustomerID"].nunique()),
    "association_rules_count": int(len(association_rules)),
    "association_method": "exact pairwise co-occurrence; not FP-Growth",
    "association_products_evaluated": int(len(selected_products))
}
with open(OUTPUT_DIR / "modeling_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print("Outputs saved to:", OUTPUT_DIR.resolve())

{
  "customers": 4334,
  "invoices": 18267,
  "product_columns": 1612,
  "selected_model": "KMeans",
  "selected_k": 2,
  "selected_features": [
    "Seg_Recency",
    "Seg_Frequency",
    "Seg_Monetary",
    "Seg_UniqueProducts"
  ],
  "sample_size_for_silhouette": 2500,
  "silhouette_sampled": 0.3989477549093732,
  "davies_bouldin": 0.9516318108379871,
  "calinski_harabasz": 3902.8560059110005,
  "cluster_sizes": {
    "0": 2582,
    "1": 1752
  },
  "top500_unique_customers": 500,
  "association_rules_count": 142,
  "association_method": "exact pairwise co-occurrence; not FP-Growth",
  "association_products_evaluated": 150
}
Outputs saved to: C:\Users\Zhivar\Desktop\OnlineShope_DataMining\modeling_outputs
